In [12]:
import os

os.environ["DOLT_HOST"] = "localhost"
import pandas as pd
import stats
import importlib

In [ ]:
importlib.reload(stats)

import yfinance as yf
import fred
import numpy as np
from curl_cffi import requests

session = requests.Session(impersonate="chrome")

predict = pd.read_csv("rs3000.csv").iloc[:, 1:]
benchmark_close = (
    yf.Ticker("SPY", session=session)
    .history(period="5y", interval="1mo")
    .Close.reset_index(drop=True)
)
rf_rate = np.float64(fred.rf_rate())
stats.value_score("MSFT", predict.loc[:, "MSFT"], benchmark_close, rf_rate)

INFO:stats:Scoring MSFT


{'ticker': 'MSFT',
 'sector': 'Technology',
 'wacc': 0.1347953574349856,
 'est_price': 24.84112084842514,
 'price': 505.82000732421875,
 'diff': -480.9788864757936,
 'score': -0.950889406332829}

In [3]:
scores_df = pd.read_csv("scores1.csv").iloc[:, 1:]
scores_df["quantile"] = pd.qcut(scores_df["score"], 5, labels=False)
scores_df.sort_values(["quantile", "score"], ascending=False)

,ticker,sector,wacc,est_price,price,diff,score,quantile
221,BIIB,Healthcare,0.036578,2640.795490,143.660004,2497.135486,17.382260,4
149,GIS,Consumer Defensive,0.045227,283.465224,62.330002,221.135222,3.547814,4
212,SYF,Financial Services,0.139648,217.542602,51.990002,165.552601,3.184316,4
106,MET,Financial Services,0.115998,251.388757,78.449997,172.938760,2.204446,4
223,DECK,Consumer Cyclical,0.155369,359.897824,120.529999,239.367825,1.985961,4
...,...,...,...,...,...,...,...,...
209,FE,Utilities,0.064817,-92.187576,39.150002,-131.337578,-3.354727,0
127,XEL,Utilities,0.062733,-185.024073,68.730003,-253.754077,-3.692042,0
226,ES,Utilities,0.067766,-225.701023,60.540001,-286.241024,-4.728130,0
111,D,Utilities,0.076647,-210.004014,55.139999,-265.144013,-4.808560,0


In [7]:
scores_df[scores_df["ticker"] == "MSFT"]

,ticker,sector,wacc,est_price,price,diff,score,quantile
1,MSFT,Technology,0.134474,111.449798,380.450012,-269.000214,-0.707058,2


In [4]:
scores_df["score"].describe()

count    248.000000
mean      -0.567571
std        1.514368
min       -5.008285
25%       -0.828613
50%       -0.683494
75%       -0.413730
max       17.382260
Name: score, dtype: float64

In [5]:
sector_groups = pd.read_csv("scores1.csv").iloc[:, 1:].groupby("sector")
for name, group in sector_groups:
    group["quantile"] = pd.qcut(group["score"], min(5, len(group)), labels=False)
    print(group["score"].describe())
    print(group.sort_values(["quantile", "score"], ascending=False))

count    12.000000
mean     -0.874753
std       0.424624
min      -1.644924
25%      -0.956842
50%      -0.806928
75%      -0.742523
max      -0.161241
Name: score, dtype: float64
    ticker           sector      wacc   est_price       price        diff  \
234    LYB  Basic Materials  0.117712   62.034641   73.959999  -11.925358   
164    NUE  Basic Materials  0.169915   76.058160  129.899994  -53.841834   
119   CTVA  Basic Materials  0.122574   23.817760   59.970001  -36.152241   
96     FCX  Basic Materials  0.203657    7.567632   35.880001  -28.312369   
163    VMC  Basic Materials  0.111310   44.753995  224.710007 -179.956012   
20     LIN  Basic Materials  0.132915   89.053701  454.070007 -365.016306   
242   STLD  Basic Materials  0.162648   22.946963  120.760002  -97.813039   
173    MLM  Basic Materials  0.121628   79.647843  464.410004 -384.762161   
188    PPG  Basic Materials  0.146984   17.123259  113.430000  -96.306742   
225    IFF  Basic Materials  0.126454  -22.385810 